In [ ]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import statsmodels.api as sm

sys.path.insert(0, '..')
from src.effects import ate

parquet_path = "../data/processed/criteo-uplift-v2.1.parquet"

In [ ]:
df1 = pd.read_parquet(parquet_path)

# Full shuffle, not a subsample: frac=1 keeps every row. Lower frac if things get slow.
df_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
visit_mean_treated, visit_mean_untreated, ate_visit, _ = ate(df_shuffled, 'visit')
print(f'Mean visit for Ad group: {visit_mean_treated:.6f}')
print(f'Mean visit for No-ad group: {visit_mean_untreated:.6f}')
print(f'ATE for visit: {ate_visit:.6f}')

conversion_mean_treated, conversion_mean_untreated, ate_conversion, _ = ate(df_shuffled, 'conversion')
print(f'Mean conversion for Ad group: {conversion_mean_treated:.6f}')
print(f'Mean conversion for No-ad group: {conversion_mean_untreated:.6f}')
print(f'ATE for conversion: {ate_conversion:.6f}')

relative_lift_visit = ate_visit / visit_mean_untreated
relative_lift_conversion = ate_conversion / conversion_mean_untreated

In [ ]:
# Bootstrap 95% interval for the conversion ATE, on a 1M-user subsample to keep it fast.
# Users are resampled with replacement within each group, so group sizes stay fixed.
boot_sample = df_shuffled.sample(n=1_000_000, random_state=42)
conversion_ad = boot_sample[boot_sample['treatment']==1]['conversion'].to_numpy()
conversion_no_ad = boot_sample[boot_sample['treatment']==0]['conversion'].to_numpy()

rng = np.random.default_rng(42)
n_boot = 1000
boot_ates = np.empty(n_boot)
for b in range(n_boot):
    resampled_ad = rng.choice(conversion_ad, size=len(conversion_ad), replace=True)
    resampled_no_ad = rng.choice(conversion_no_ad, size=len(conversion_no_ad), replace=True)
    boot_ates[b] = resampled_ad.mean() - resampled_no_ad.mean()

boot_low, boot_high = np.percentile(boot_ates, [2.5, 97.5])

# Normal interval on the same subsample, for comparison.
ate_conversion_sub = conversion_ad.mean() - conversion_no_ad.mean()
SE_conversion_sub = np.sqrt(conversion_ad.var(ddof=1)/len(conversion_ad) + conversion_no_ad.var(ddof=1)/len(conversion_no_ad))

print(f'ATE for conversion (1M subsample): {ate_conversion_sub:.6f}')
print(f'Bootstrap 95% CI ({n_boot} resamples): [{boot_low:.6f}, {boot_high:.6f}]')
print(f'Normal 95% CI (same subsample):    [{ate_conversion_sub - 1.96*SE_conversion_sub:.6f}, {ate_conversion_sub + 1.96*SE_conversion_sub:.6f}]')

In [ ]:
# Robustness check: covariate-adjusted ATE. Regress conversion on treatment + f0..f11 (OLS, robust SEs).
# If the treatment coefficient is close to the simple ATE, the small feature imbalance does not matter.
X = sm.add_constant(df_shuffled[['treatment'] + [f'f{i}' for i in range(12)]].astype('float64'))
adjusted_model = sm.OLS(df_shuffled['conversion'].astype('float64'), X).fit(cov_type='HC1')
ate_conversion_adjusted = adjusted_model.params['treatment']
adjusted_low, adjusted_high = adjusted_model.conf_int().loc['treatment']

print(f'Simple ATE for conversion:   {ate_conversion:.6f}')
print(f'Adjusted ATE for conversion: {ate_conversion_adjusted:.6f}, 95% CI [{adjusted_low:.6f}, {adjusted_high:.6f}]')
print(f'Adjusted vs simple: {(ate_conversion_adjusted - ate_conversion) / ate_conversion * 100:+.1f}%')

In [ ]:
# Phase 2 results table: one row per outcome.
# The conversion model is already fit above, so only visit gets a new fit here.
results = []
for outcome in ['visit', 'conversion']:
    ad_rate, no_ad_rate, outcome_ate, se = ate(df_shuffled, outcome)
    if outcome == 'conversion':
        adjusted_ate = ate_conversion_adjusted
    else:
        adjusted_ate = sm.OLS(df_shuffled[outcome].astype('float64'), X).fit(cov_type='HC1').params['treatment']
    results.append({
        'outcome': outcome,
        'no-ad rate': no_ad_rate,
        'ad rate': ad_rate,
        'ATE': outcome_ate,
        '95% CI': f'[{outcome_ate - 1.96*se:.6f}, {outcome_ate + 1.96*se:.6f}]',
        'relative lift': f'{outcome_ate / no_ad_rate:.1%}',
        'adjusted ATE': adjusted_ate,
    })
results_table = pd.DataFrame(results).set_index('outcome')
print(results_table.to_string(float_format='{:.6f}'.format))
results_table.to_csv('../figures/phase2_results.csv')

In [ ]:
# Feature checks: conversion rate across each feature's deciles, split by treatment.
# Deciles are by rank, so features with one dominant value still get 10 equal-sized groups.
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    bins = pd.qcut(df_shuffled[f'f{i}'].rank(method='first'), 10, labels=False)
    rates = df_shuffled.groupby([bins, 'treatment'])['conversion'].mean().unstack()
    ax.plot(range(len(rates)), rates[1] * 100, marker='o', label='Ad group')
    ax.plot(range(len(rates)), rates[0] * 100, marker='o', label='No-ad group')
    ax.set_title(f'f{i}')
    ax.set_xlabel('Feature decile (low to high)')
    ax.set_ylabel('Conversion rate (%)')
    ax.grid(alpha=0.3)
axes[0, 0].legend()
fig.suptitle('Conversion rate by feature decile: Ad group vs No-ad group')
plt.tight_layout()
plt.savefig('../figures/feature_conversion_by_treatment.png', dpi=150)
plt.show()